
# Optimizing Neural Networks

Training a neural network is an optimization problem. We want to reach the minimum error as fast as possible without getting stuck. Basic Gradient Descent is often too slow or unstable for deep networks. To fix this, we use advanced **Optimization** and **Regularization** techniques.

## Weight Initialization: Starting Right

If we initialize all weights to Zero, every neuron learns the same thing (symmetry problem). The network becomes a linear model. If we initialize them too large, gradients explode. Too small, gradients vanish.

**Strategies**:

-   **He Initialization**: Best for ReLU networks. Keeps the variance of activations constant across layers.
-   **Xavier (Glorot) Initialization**: Best for Sigmoid/Tanh networks.

## Optimizers: Smarter Descent

Standard SGD is like walking down a mountain blindfolded. Advanced optimizers add "momentum" or adaptive steps.

| Optimizer          | Mechanism                                               | Analogy                                                 |
|------------------ |------------------------------------------------------- |------------------------------------------------------- |
| **SGD + Momentum** | Accumulates past gradients to smooth out the path.      | A heavy ball rolling downhill (builds speed).           |
| **RMSProp**        | Adapts the learning rate for each parameter separately. | Slows down on steep slopes, speeds up on flat plains.   |
| **Adam**           | Combines Momentum + RMSProp. The default choice.        | A smart ball that adapts speed and direction perfectly. |

## Regularization: Fighting Overfitting

Deep networks are prone to memorizing data. We need techniques to force them to generalize.

### Dropout

Randomly "kill" (set to zero) a percentage of neurons during training. This prevents neurons from co-adapting and relying too much on specific features. It forces the network to be redundant and robust.

### Batch Normalization

Normalizes the inputs of each layer to have mean 0 and variance 1 **during training**.

-   Stabilizes training (allows higher learning rates).
-   Reduces sensitivity to initialization.

### Early Stopping

Monitor the Validation Loss. If it stops improving (or starts getting worse), stop training immediately. This prevents the model from overfitting in the late stages.

## Practical Demonstration: Adam vs SGD

We will train a network on the "Digits" dataset using standard SGD versus Adam to see the speed difference.

### Setup Data

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Load and Scale
digits = load_digits()
X, y = digits.data, digits.target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

### Training Comparison

We train two identical MLPs, changing only the `solver`.

In [ ]:
from sklearn.neural_network import MLPClassifier

# SGD Model
mlp_sgd = MLPClassifier(hidden_layer_sizes=(64, 32), 
                        solver='sgd', 
                        learning_rate_init=0.01, 
                        max_iter=200, 
                        random_state=42)

# Adam Model
mlp_adam = MLPClassifier(hidden_layer_sizes=(64, 32), 
                         solver='adam', 
                         learning_rate_init=0.01, 
                         max_iter=200, 
                         random_state=42)

mlp_sgd.fit(X_train, y_train)
mlp_adam.fit(X_train, y_train)

print(f"SGD Accuracy:  {mlp_sgd.score(X_test, y_test):.3f}")
print(f"Adam Accuracy: {mlp_adam.score(X_test, y_test):.3f}")

### Visualizing Convergence Speed

We plot the Loss Curves. Adam usually drops much faster.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(mlp_sgd.loss_curve_, label='SGD', color='red', linestyle='--')
plt.plot(mlp_adam.loss_curve_, label='Adam', color='blue')
plt.title("Convergence Speed: Adam vs SGD")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Practical Demonstration: Early Stopping

We will force a network to overfit and see if Early Stopping saves us.

In [ ]:
# Model WITH Early Stopping (default in sklearn)
# early_stopping=True sets aside 10% of training data for validation
mlp_early = MLPClassifier(hidden_layer_sizes=(100,), 
                          max_iter=500, 
                          early_stopping=True, 
                          validation_fraction=0.1,
                          n_iter_no_change=10, 
                          random_state=42)

mlp_early.fit(X_train, y_train)

print(f"Stopped after {mlp_early.n_iter_} iterations.")
print(f"Best Validation Score: {mlp_early.best_validation_score_:.3f}")

# Plot the Validation Curve
plt.figure(figsize=(8, 5))
plt.plot(mlp_early.loss_curve_, label='Training Loss')
plt.plot(mlp_early.validation_scores_, label='Validation Score (Accuracy)')
plt.title("Early Stopping Monitor")
plt.xlabel("Iterations")
plt.legend()
plt.show()

## Exercises

### The Effect of Batch Normalization (Simulated)

Scikit-learn's MLP doesn't have a simple "Batch Norm" flag (it's built into the solvers implicitly or requires libraries like PyTorch/TensorFlow). However, we can simulate the **need** for it by using unscaled data.

1.  Train an MLP on `X_train` (Scaled).
2.  Train an MLP on `X` (Unscaled raw data).
3.  Compare the loss curves.

### Regularization Strength (Alpha)

The `alpha` parameter controls L2 Regularization (Weight Decay).

-   Train a model with `alpha=0.0001` (Default).
-   Train a model with `alpha=1.0` (High Regularization).
-   Compare accuracies.

**Result**: High alpha should slightly reduce Training Accuracy (preventing overfitting) but might maintain or improve Test Accuracy.

## Summary

1.  **Adam**: generally the best default optimizer. Fast and robust.
2.  **Early Stopping**: The easiest way to prevent overfitting without tuning hyperparameters.
3.  **Normalization**: Neural Networks fail without scaled data (Batch Norm automates this deep inside layers).